# Notebook 6 (optional) - Find irrigation ponds with a Random Forest

<a target="_blank" href="https://colab.research.google.com/github/khouakhi/UMP_EO_training/blob/main/notebooks/06_optional_random_forest_land_cover.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


## Objective

This notebook presents a **self-contained** supervised workflow: collect marker points on **pond** and **non-pond** pixels, train a **binary Random Forest** (**pond = 1**, **not pond = 0**), then map likely ponds and count separate patches (each patch -> **one centroid** on the map).

## Data and masks

- **Satellite:** a **cloud-masked mean Sentinel-2 image** from [**Sentinel-2 SR Harmonized**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) over **March-April 2026**. We keep a relatively high scene cloud threshold, then mask cloud/shadow pixels per image and take the mean for stable full-AOI coverage.
- **Where we run the model:** we remove **urban** pixels with [**ESA WorldCover v200**](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200) (class **50**) and keep only gentle terrain from [**SRTM DEM**](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003) using a user-set **`SLOPE_MAX_DEG`** threshold.

The classifier uses **scaled reflectance** plus **NDVI** and **NDWI**, the same spectral contrast that makes ponds look **dark** in false colour (**B8, B4, B3** on screen).

**Time tip:** about **30–40 minutes**. **Prerequisite:** notebooks **00** and **01** (Earth Engine + AOI).


In [ ]:
# Install packages (Colab often needs a fresh install each session)
!pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# If you run this notebook locally, run `ee.Authenticate()` once before `ee.Initialize`.
# If the Colab pop-up fails, try: ee.Authenticate(auth_mode="colab")
# Mapping notebooks use `Map.add_basemap("SATELLITE")` so you always have photo context under EE layers.


In [ ]:
# Connect to Google Earth Engine using your cloud project ID.
# Set this to your own Google Earth Engine cloud project ID before running.
EE_PROJECT = "YOUR_GEE_PROJECT_ID"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialised with project:", EE_PROJECT)


In [ ]:
# Study area: Moulouya basin - HydroSHEDS level-8 hydrological unit (WWF)
# Dataset: WWF/HydroSHEDS/v1/Basins/hybas_8 - use the same HYBAS_ID in every notebook for consistency.

HYBAS_ID = 1080030220

MOULOUYA_BASIN_H08 = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8").filter(
    ee.Filter.eq("HYBAS_ID", HYBAS_ID)
)

# Geometry used for clips, filterBounds, reduceRegion, etc.
LOWER_MOULOUYA_AOI = MOULOUYA_BASIN_H08.geometry()

# Extended winter–spring wet season (December–April), named by the April that closes the window.


def wet_season_filter_dates(april_year: int) -> tuple[str, str]:
    # Returns filterDate(start, end) with end exclusive; April is fully included.
    start = f"{april_year - 1}-12-01"
    end = f"{april_year}-05-01"
    return (start, end)


## Build one consistent Sentinel-2 image (mean composite)

We use **March-April 2026**, allow scene metadata cloud up to **40 %**, mask cloud/shadow pixels with **SCL**, then take the **mean**. This keeps code simple and usually gives full basin coverage.


In [ ]:
YEAR_SCENE = 2026
CLOUD_MAX = 40
N_MEAN = 20


def mask_s2_clouds(img: ee.Image) -> ee.Image:
    scl = img.select("SCL")
    bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
    return img.updateMask(bad.Not())


s2_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(LOWER_MOULOUYA_AOI)
    .filterDate(f"{YEAR_SCENE}-03-01", f"{YEAR_SCENE}-05-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

n_scenes = int(s2_col.size().getInfo())
if n_scenes < 1:
    raise RuntimeError("No Sentinel-2 scenes found for March-April 2026.")

scene = (
    s2_col.limit(N_MEAN)
    .map(mask_s2_clouds)
    .select(["B2", "B3", "B4", "B8", "B11"])
    .mean()
    .clip(LOWER_MOULOUYA_AOI)
)
print("Scenes used in mean:", min(N_MEAN, n_scenes))


## Quick check - composite coverage

Run this map now to confirm there are no tile-edge gaps before moving on.


In [ ]:
SCALE = 1 / 10_000
vis_rgb = {"bands": ["B8", "B4", "B3"], "min": 0.06, "max": 0.45, "gamma": 1.05}
aoi_vis = {"color": "red"}
rgb_preview = scene.multiply(SCALE).select(["B8", "B4", "B3"])

Map_scene = geemap.Map()
Map_scene.add_basemap("SATELLITE")
Map_scene.centerObject(LOWER_MOULOUYA_AOI, 9)
Map_scene.addLayer(rgb_preview, vis_rgb, "False colour preview (Mar-Apr mean)")
Map_scene.addLayer(LOWER_MOULOUYA_AOI, aoi_vis, "AOI", opacity=0.22)
Map_scene.add_layer_control()
Map_scene


## Work mask: remove urban, then apply slope

We exclude **urban** using [**ESA WorldCover v200**](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200) class **50**, then keep only gentle terrain from [**SRTM DEM**](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003).

You can tune **`SLOPE_MAX_DEG`** to widen or narrow the final mask.


In [ ]:
# User options
SLOPE_MAX_DEG = 15

wc = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .filterDate("2021-01-01", "2022-01-01")
    .first()
    .select("Map")
    .clip(LOWER_MOULOUYA_AOI)
)
urban = wc.eq(50)

dem = ee.Image("USGS/SRTMGL1_003").clip(LOWER_MOULOUYA_AOI)
slope_deg = ee.Terrain.slope(dem)
gentle = slope_deg.lt(SLOPE_MAX_DEG)

work_mask = urban.Not().And(gentle).rename("work").byte()


## Map - mask components and final work mask

- **Grey**: urban (excluded)
- **Green**: gentle slope (kept)
- **Orange**: final mask used by the Random Forest


In [ ]:
urban_vis = {"palette": ["#969696"], "opacity": 0.65}
gentle_vis = {"palette": ["#74c476"], "opacity": 0.35}
work_mask_vis = {"palette": ["#fecc5c"], "opacity": 0.85}
aoi_vis = {"color": "red"}

Map_plain = geemap.Map()
Map_plain.add_basemap("SATELLITE")
Map_plain.centerObject(LOWER_MOULOUYA_AOI, 9)
Map_plain.addLayer(urban.selfMask(), urban_vis, "Urban (WorldCover class 50)")
Map_plain.addLayer(gentle.selfMask(), gentle_vis, f"Gentle slope < {SLOPE_MAX_DEG} deg")
Map_plain.addLayer(work_mask.selfMask(), work_mask_vis, "Final work mask")
Map_plain.addLayer(LOWER_MOULOUYA_AOI, aoi_vis, "AOI", opacity=0.22)
Map_plain.add_layer_control()
Map_plain


## Spectral stack for the Random Forest

Bands are scaled with **`SCALE = 1 / 10_000`**. Indices help separate **open water** from **bright vegetation** in the plains.


In [ ]:
SCALE = 1 / 10_000


def stack_for_rf(img: ee.Image) -> ee.Image:
    x = img.multiply(SCALE)
    ndvi = x.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = x.normalizedDifference(["B3", "B8"]).rename("NDWI")
    return x.addBands(ndvi).addBands(ndwi)


stack_img = stack_for_rf(scene)
bands = ["B2", "B3", "B4", "B8", "B11", "NDVI", "NDWI"]

vis_rgb = {"bands": ["B8", "B4", "B3"], "min": 0.06, "max": 0.45, "gamma": 1.05}
work_mask_vis = {"palette": ["#ffeda0"], "opacity": 0.35}
aoi_vis = {"color": "red"}
rgb = scene.multiply(SCALE).select(["B8", "B4", "B3"])


def safe_clear_drawn(map_widget):
    # Clear user drawings; tolerates geemap/ipyleaflet layer list desynchronisation.
    try:
        map_widget.remove_drawn_features()
    except Exception:
        dc = getattr(map_widget, "_draw_control", None)
        if dc is None:
            return
        try:
            lyr = dc.layer
            if lyr is not None and lyr in map_widget.layers:
                map_widget.remove_layer(lyr)
        except Exception:
            pass
        dc.geometries = []
        dc.properties = []
        dc.last_geometry = None
        dc.layer = None
        ee_layers = getattr(map_widget, "ee_layers", None)
        if ee_layers is not None:
            ee_layers.pop("Drawn Features", None)
        if hasattr(dc, "data"):
            dc.data = []
            dc.send_state(key="data")


Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map.centerObject(LOWER_MOULOUYA_AOI, 11)
Map.addLayer(rgb, vis_rgb, "False colour (Mar-Apr mean)")
Map.addLayer(work_mask.selfMask(), work_mask_vis, "Work mask (not urban, gentle slope)")
Map.addLayer(LOWER_MOULOUYA_AOI, aoi_vis, "AOI", opacity=0.12)
Map.add_layer_control()
Map


## 1. Marker points on **ponds** (`class = 1`)

Use the **point / marker** tool. Place **≥ 8** markers on **dark pond** pixels inside the yellow plains mask, then run the next cell.


In [ ]:
pond_markers = ee.FeatureCollection(Map.draw_features).map(lambda f: f.set("class", 1))
if pond_markers.size().getInfo() < 8:
    raise ValueError("Add at least 8 pond markers, then re-run.")
safe_clear_drawn(Map)
print("Pond markers:", pond_markers.size().getInfo())


## 2. Markers on land that is **not** a pond (`class = 0`)

Place **≥ 8** markers on **crop**, **soil**, or **tracks** (clearly **not** open water) within the same yellow mask, then run the next cell.


In [ ]:
other_markers = ee.FeatureCollection(Map.draw_features).map(lambda f: f.set("class", 0))
if other_markers.size().getInfo() < 8:
    raise ValueError("Add at least 8 non-pond markers, then re-run.")
safe_clear_drawn(Map)
print("Non-pond markers:", other_markers.size().getInfo())


## 3. Train, classify, vectorise, count

`sampleRegions` reads spectra at **10 m** under your markers. We train on **all labelled points** and predict directly. Predictions are limited to **`work_mask`**. **`reduceToVectors`** builds polygons; **centroids** are one point per patch for mapping and **patch count**.


In [ ]:
train_fc = pond_markers.merge(other_markers)

training = stack_img.select(bands).sampleRegions(
    collection=train_fc,
    properties=["class"],
    scale=10,
    geometries=True,
)
print("Training samples:", int(training.size().getInfo()))

classifier = ee.Classifier.smileRandomForest(numberOfTrees=80, seed=5).train(
    features=training,
    classProperty="class",
    inputProperties=bands,
)

pred = stack_img.select(bands).classify(classifier).rename("label").byte()
pond_mask = pred.eq(1).updateMask(work_mask).rename("pond")

AREA_MIN_M2 = 300.0
vectors = pond_mask.selfMask().reduceToVectors(
    geometry=LOWER_MOULOUYA_AOI,
    scale=20,
    geometryType="polygon",
    eightConnected=False,
    maxPixels=1e10,
    tileScale=4,
    labelProperty="pond",
)
with_area = vectors.map(lambda f: f.set("area_m2", f.geometry().area(maxError=10)))
pond_patches = with_area.filter(ee.Filter.gte("area_m2", AREA_MIN_M2))
pond_points = pond_patches.map(
    lambda f: ee.Feature(f.geometry().centroid(10), {"area_m2": f.get("area_m2")})
)

n_patches = int(pond_patches.size().getInfo())
print("Predicted pond patches (minimum area filter):", n_patches)


## Map - ponds, patch centres, training markers

**Magenta:** predicted **patch centres**. **Cyan / yellow:** your **pond / not-pond** training markers. Compare **RF pond mask** and **plains mask** against false colour.


In [ ]:
train_pond = train_fc.filter(ee.Filter.eq("class", 1))
train_other = train_fc.filter(ee.Filter.eq("class", 0))
plains_mask_vis = {"palette": ["#ffeda0"], "opacity": 0.25}
pond_mask_vis = {"palette": ["#225ea8"], "opacity": 0.45}
pond_points_vis = {"color": "#ff00ff", "pointRadius": 5}
train_pond_vis = {"color": "#00ffff", "pointRadius": 6}
train_other_vis = {"color": "#ffff00", "pointRadius": 6}
aoi_vis = {"color": "red"}

Map2 = geemap.Map()
Map2.add_basemap("SATELLITE")
Map2.centerObject(LOWER_MOULOUYA_AOI, 11)
Map2.addLayer(rgb, vis_rgb, "False colour (April mean)")
Map2.addLayer(work_mask.selfMask(), plains_mask_vis, "Plains mask", shown=False)
Map2.addLayer(pond_mask.selfMask(), pond_mask_vis, "RF pond mask")
Map2.addLayer(pond_points, pond_points_vis, "Pond patch centres")
Map2.addLayer(train_pond, train_pond_vis, "Training: pond")
Map2.addLayer(train_other, train_other_vis, "Training: not pond")
Map2.addLayer(LOWER_MOULOUYA_AOI, aoi_vis, "AOI", opacity=0.1)
Map2.add_layer_control()
Map2


## Reflect

1. Why does **patch count** depend on both the **model** and the **`reduceToVectors` scale**?
2. If river lines still leak through, how would you add a **manual river centreline buffer** before vectorising, without changing the Random Forest training code?

**Congratulations** - you have finished the optional **Random Forest** exercise for this module.
